Import libraries, set date range for analysis period, and initialize a `Sources` class object - it contains all the Sources/index name and the corresponding tickers that we need. We also initialize `data_frames` to compile all DataFrames.

In [ ]:
import requests
import os
import sys
import time
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime
from fredapi import Fred
from dotenv import load_dotenv
from openpyxl.styles import PatternFill, Font, Alignment

sys.path.append('..')
from modules.source import Sources, RAW_DATA_PATH, PROCESSED_DATA_PATH, OUTPUT_FOLDER_PATH, parquet_monthly, parquet_daily
from modules.helpers import clean_api_response, parse_10y_entry, combine_trading_days

START = datetime(2015, 1, 1)
END = datetime(2026, 8, 25)
   
load_dotenv("../.env")
FRED_API_KEY = os.getenv('FRED_API_KEY')
if FRED_API_KEY is None: print('Enter your FRED API key in .env')
fred = Fred(FRED_API_KEY)

US_indices = [Sources.UST_10Y, Sources.VIX, Sources.USDMYR, Sources.brent_oil, Sources.DXY, Sources.palm_oil_global, Sources.FFR_midpoint]
MY_indices = [Sources.KLCI, Sources.MGS_10Y, Sources.financials, Sources.plantation, Sources.reits, Sources.technology, Sources.energy, Sources.industrial_products]
data_frames: dict[pd.DataFrame] = {}

Since these operations are identical across all tickers, from here on we only note material deviations from this structure.
1. **Fetch** - pull data from FRED
2. **Save** - write the raw response to `.csv`
3. **Process** - use `clean_api_response` to standardize data format, and adjust to add back publication lag (in this case, 2 months) 
4. **Add to `data_frames` object**

We convert the raw data to DataFrame before saving it purely for pipeline consistency. Saving it as a Series instead would produce an identical CSV.



In [ ]:
series = fred.get_series_first_release(Sources.palm_oil_global.ticker)

df = clean_api_response(series, Sources.palm_oil_global)
df.index = pd.to_datetime(df.index)

df.to_csv(f"{RAW_DATA_PATH}/{Sources.palm_oil_global}.csv")

df = df.shift(2, freq="BME")
df = df.loc[START:END]
data_frames[Sources.palm_oil_global] = df

We take the midpoint of the FFR range, and adjust for the 1 day reporting delay after its implementation day (shift it back to the implementation/announcement date).

In [ ]:
series_upper = fred.get_series('DFEDTARU')
series_lower = fred.get_series('DFEDTARL')

df = pd.concat([series_upper, series_lower], axis=1, join='inner')
df.columns = ['Upper', 'Lower']
df['Midpoint'] = (df['Upper'] + df['Lower']) / 2
df.index.name = 'date'
df.to_csv(f"{RAW_DATA_PATH}/{Sources.FFR_midpoint}.csv")

df = df[['Midpoint']].rename(columns={'Midpoint': Sources.FFR_midpoint})
df = df.shift(-1)
df = df.loc[START:END]
data_frames[Sources.FFR_midpoint] = df

We pull the data from yfinance, and use the daily closing price as our price data.

In [ ]:
yahoo_stocks = [Sources.UST_10Y, Sources.VIX, Sources.USDMYR, Sources.DXY, Sources.brent_oil, Sources.KLCI]

for name in yahoo_stocks:
   df = yf.download(name.ticker, START, END, progress=False)
   df.to_csv(f'{RAW_DATA_PATH}/{name}.csv')

   try:
      df = clean_api_response(df, name)
   except ValueError as e:
      print(f"  FAILED - {name}: {e}")
      continue

   print(f"{name} ({name.ticker}) - saved {len(df)} rows")
   df = df[['Close']].rename(columns={'Close': name})
   data_frames[name] = df

For CPI inflation by DOSM, we narrow it down to overall division only and YoY inflation as our primary series. And undo the ~2 month of publication lag.

One thing to note: core CPI data is only available from 2018 onwards, so YoY (needing a 12-month lookback) is only available from 2019 onwards - this affects CPI-related analysis specifically, not the full project's date range.

In [ ]:
df_cpi = pd.read_parquet('https://storage.dosm.gov.my/cpi/cpi_2d_core_inflation.parquet')
df_cpi.to_csv(f'{RAW_DATA_PATH}/CPI.csv')

df_cpi = (
   df_cpi
   .assign(date=pd.to_datetime(df_cpi['date']))
   .set_index('date')
   .query("division == 'overall'")
   .drop(columns=['division', 'inflation_mom'])
   .shift(2, freq="BME")
   .loc[START:END]
   .dropna()
   .rename(columns={'inflation_yoy': Sources.cpi_inflation_yoy})
)

data_frames[Sources.cpi_inflation_yoy] = df_cpi

As with core CPI, for OPR, we set `date` as the index, then sort it before saving. The year-by-year fetch doesn't guarantee chronological order.

Post-save, we drop `year` (redundant once `date` is the index) and `change_in_opr` (we'll reconstruct this ourselves downstream), and rename `new_opr_level` to `OPR`.

In [ ]:
headers = {'Accept': 'application/vnd.BNM.API.v1+json'}

records = []
for year in range(START.year, END.year + 1):
   resp = requests.get(f'https://api.bnm.gov.my/public/opr/year/{year}', headers=headers)
   records.extend(resp.json()['data'])

df_opr = pd.DataFrame(records)
df_opr['date'] = pd.to_datetime(df_opr['date'])
df_opr = df_opr.set_index('date')
df_opr = df_opr.sort_index()

df_opr = df_opr[~df_opr.index.duplicated(keep='last')]
df_opr.to_csv(f'{RAW_DATA_PATH}/{Sources.OPR}.csv')

df_opr = (
   df_opr.loc[START:END]
   .drop(columns=['year', 'change_in_opr'])
   .rename(columns={'new_opr_level': Sources.OPR})
)

data_frames[Sources.OPR] = df_opr

BNM only provides daily MGS yield per date request, we will have to make ~4,000 requests. With 0.3 seconds delay per request (to prevent API request overload), it takes roughly 30 minutes to fetch all data. 

In [ ]:
BASE_URL = "https://api.bnm.gov.my/public/gov-sec-yield"
HEADERS = {"Accept": "application/vnd.BNM.API.v1+json"}

business_days = pd.bdate_range(START, END)  # Mon-Fri only; holidays still queried but expected empty

records = []
no_data_days = []   # likely public holidays
failed_days = []    # genuine API/network failures - worth re-running individually

for year, year_group in business_days.to_series().groupby(business_days.year):
   year_records = 0
   for d in year_group:
      date_str = d.strftime("%Y-%m-%d")
      resp = requests.get(BASE_URL, headers=HEADERS, params={"date": date_str}, timeout=30)

      if resp.status_code != 200:
         try:
            err = resp.json()
            print(f"[{date_str}] Failed: {err.get('code', resp.status_code)} - {err.get('message', 'Unknown error')}")
         except ValueError:
            print(f"[{date_str}] Failed: HTTP {resp.status_code} - {resp.text[:200]}")
         failed_days.append(date_str)
         time.sleep(0.3)
         continue

      payload = resp.json()
      entry = parse_10y_entry(payload, date_str)
      if entry is not None:
         records.append(entry)
         year_records += 1
      else:
         no_data_days.append(date_str)

      time.sleep(0.3)

   print(f"[{year}] {year_records} of {len(year_group)} business days have a 10Y entry")

df = pd.DataFrame(records).sort_values("date").reset_index(drop=True)
df.to_csv(f"{RAW_DATA_PATH}/{Sources.MGS_10Y}.csv", index=False)

print(f"\nTotal: {len(df)} daily 10Y observations")
print(f"Business days with no 10Y entry: {len(no_data_days)}")
print(f"Failed API calls: {len(failed_days)}")
if failed_days:
   print("Failed dates (re-run these individually):", failed_days)

df = df.drop(columns=['tot_vol', 'maturity_month', 'maturity_year', 'daily_change']).set_index('date')
df = df.rename(columns={'yield_close': Sources.MGS_10Y})
df.index = pd.to_datetime(df.index)
data_frames[Sources.MGS_10Y] = df

In [ ]:
df = pd.read_csv(f"{RAW_DATA_PATH}/{Sources.MGS_10Y}.csv")
df = df.drop(columns=["tot_vol", "maturity_month", "maturity_year", "daily_change"]).set_index("date")
df = df.rename(columns={"yield_close": Sources.MGS_10Y})
df.index = pd.to_datetime(df.index)
data_frames[Sources.MGS_10Y] = df

For each sector, we aggregate 2-3 large, widely-held constituents, save to CSV, then reconstruct its equal-weighted index. For plantation, we trim data from before 2017-11-30 (SD Gurthie Berhad hasn't listed yet) or else it would produce price-level break.

In [ ]:
sector_constituents = {
   Sources.financials: ['1155.KL', '1023.KL', '1295.KL'],   # Maybank, CIMB, Public Bank
   Sources.plantation: ['5285.KL', '1961.KL', '2445.KL'],   # Sime Darby Plantation, IOI Corp, KLK
   Sources.reits:      ['5227.KL', '5176.KL', '5235SS.KL'], # IGB REIT, Sunway REIT, KLCCP Stapled
   Sources.technology: ['0166.KL', '0097.KL', '0128.KL'],   # Inari Amertron, ViTrox, Frontken
   Sources.energy:     ['6033.KL', '5183.KL', '5681.KL'],   # PetGas, PetChem, Petronas Dagangan
   Sources.industrial_products: ['8869.KL', '7113.KL'],     # Press Metal, Top Glove
}

for sector, tickers in sector_constituents.items():
   print(f"\n=== {sector.upper()} ===")
   closing_price_series: dict[pd.Series] = {}
   
   for t in tickers:
      df = yf.download(t, START, END, progress=False)
      try:
         df = clean_api_response(df, t)
      except ValueError as e:
         print(f"  FAILED - {t}: {e}")
         continue

      closing_price_series[t] = df['Close'] # stored as series
      print(f"  OK - {t} ({len(df)} rows)")

   sector_df = pd.DataFrame(closing_price_series)
   sector_df.to_csv(f'{RAW_DATA_PATH}/sectors/{sector}.csv')
   sector_df.index.name = 'date'

   if sector is Sources.plantation:
      cutoff = datetime(2017, 11, 30) # 5285.KL (SD Guthrie Berhad) date of being listed
      sector_df = sector_df.loc[cutoff:]

   sector_df[sector] = sector_df.ffill().pct_change().mean(axis=1)
   sector_df[sector] = 100 * (1 + sector_df[sector].fillna(0)).cumprod()
   data_frames[sector] = sector_df[[sector]]

Now we forward fill all data, adjust for timezone difference.

In [ ]:
monthly_series:     list[str] = [Sources.palm_oil_global, Sources.FFR_midpoint, Sources.OPR, Sources.cpi_inflation_yoy]
daily_freq_indices: list[str] = [name for name in US_indices + MY_indices if name not in monthly_series]
all_trading_days:   pd.Index  = combine_trading_days([data_frames[name] for name in daily_freq_indices])
US_trading_days:    pd.Index  = combine_trading_days([data_frames[name] for name in US_indices if name is not Sources.FFR_midpoint])
original_dates: dict[pd.Index] = {n: df.index for n, df in data_frames.items()}

for n in monthly_series: # Data could be published on non-trading days. Fill all days then trim to trading days.
   data_frames[n] = (
      data_frames[n]
      .reindex(pd.date_range(START, END, freq="D"))
      .ffill()
      .reindex(index=all_trading_days)
   )

for n in daily_freq_indices:
   data_frames[n] = data_frames[n].reindex(index=all_trading_days).ffill()

for n in US_indices: # timezone adjustment, not needed for Palm Oil since its monthly published data
   if n is Sources.palm_oil_global: continue
   data_frames[n] = data_frames[n].shift(1)

all_trading_days = all_trading_days.delete(0) # delete the 1st index where there is no data
for n, df in data_frames.items():
   data_frames[n] = df.reindex(index=all_trading_days)

Create MultiIndex DataFrames to store whether a data is forward filled in the `is_ffilled` column, and combine all DataFrames.

In [ ]:
dfs_price_bool: list[pd.DataFrame] = []

for n, df in data_frames.items():
   df_bool = pd.DataFrame({"is_ffilled": ~all_trading_days.isin(original_dates[n])}, index=all_trading_days)
   df_bool = df_bool.where(df.notna().values)

   df = df.rename(columns={n: "price"})
   dfs_price_bool.append(pd.concat([df, df_bool], axis=1, keys=[n, n]))

df_combined_multiindex = pd.concat(dfs_price_bool, axis=1)

**Last step**: Resample for monthly data, export to parquet (processed data folder) and Excel (output folder), and apply gray shaded formatting to all forward-filled values in Excel. 

In [ ]:
df_combined_multiindex.to_parquet(f"{PROCESSED_DATA_PATH}/{parquet_daily}")
monthly_multiindex = df_combined_multiindex.resample("ME").last()
monthly_multiindex.to_parquet(f"{PROCESSED_DATA_PATH}/{parquet_monthly}")

prices = df_combined_multiindex.xs("price", axis=1, level=1)
flags  = df_combined_multiindex.xs("is_ffilled", axis=1, level=1)
monthly_prices = prices.resample("ME").last()
monthly_flags  = flags.resample("ME").last()

with pd.ExcelWriter(f"{OUTPUT_FOLDER_PATH}/DAILY_MASTER.xlsx", "openpyxl") as writer:
   prices.index = prices.index.date # remove timestamp
   prices.to_excel(writer)
   ws = writer.book.active

   for r, c in zip(*np.where(flags.values == True)): 
      cell = ws.cell(row=r + 2, column=c + 2) # adjust for 1-indexed cell + header row/column
      cell.fill, cell.font = PatternFill("solid", "F3F4F6"), Font(color="6B7280")

   for cell in ws[1]: # adjust column width to avoid text being clipped or column being too wide
      ws.column_dimensions[cell.column_letter].width = len(str(cell.value or "")) + 3
      if len(str(cell.value or "")) > 8:
         ws.column_dimensions[cell.column_letter].width = len(str(cell.value)) 
      cell.alignment = Alignment(horizontal="right")
   ws.column_dimensions["A"].width = 11 # date column

with pd.ExcelWriter(f"{OUTPUT_FOLDER_PATH}/MONTHLY_MASTER.xlsx", "openpyxl") as writer:
   monthly_prices.index = monthly_prices.index.date
   monthly_prices.to_excel(writer)
   ws = writer.book.active

   for r, c in zip(*np.where(monthly_flags.values == True)):
      cell = ws.cell(row=r + 2, column=c + 2)
      cell.fill, cell.font = PatternFill("solid", "F3F4F6"), Font(color="6B7280")

   for cell in ws[1]:
      ws.column_dimensions[cell.column_letter].width = len(str(cell.value or "")) + 3
      if len(str(cell.value or "")) > 8:
         ws.column_dimensions[cell.column_letter].width = len(str(cell.value)) 
      cell.alignment = Alignment(horizontal="right")
   ws.column_dimensions["A"].width = 11